# Mesh Tour — One Captured Attack, End to End

This notebook walks through one captured attack chain in the
Quantum Defense Mesh. Every cell calls the live backend over HTTP
so you can run it against `http://localhost:8000` and watch the
story unfold on the `/mesh` dashboard at the same time.

Assumes `python main.py` is running on port 8000.

In [ ]:
import json, time, hashlib, secrets
import httpx

BASE = "http://localhost:8000"
c = httpx.Client(base_url=BASE, timeout=10.0)

def show(label, obj):
    print(f"\n=== {label} ===")
    print(json.dumps(obj, indent=2, default=str)[:1200])

## Step 1 — friend opens a share-link

The operator runs `POST /mesh/share/issue` and gets a URL.
We mint one here and pretend to be the friend firing one
attack from the browser. The mesh fingerprints the visitor
by `X-Forwarded-For` + UA + the share_id.

In [ ]:
# 1. Operator issues a share token.
share = c.post("/mesh/share/issue", json={"ttl_seconds": 3600}).json()
show("share issued", share)

# 2. Friend opens the link in their browser. The HTML form POSTs
#    back to /mesh/share/{share_id}/{token}/fire. We simulate that.
xff = "203.0.113.99"  # pretend this is the friend's egress IP
ua = "Mozilla/5.0 (friend's laptop)"
fire = c.post(
    f"/mesh/share/{share['share_id']}/{share['token']}/fire?exp={share['expires_at']}",
    data={"payload": "GET /.env HTTP/1.1", "user_agent": ua, "service": "http"},
    headers={"x-forwarded-for": xff},
)
show("friend attack ingested", fire.json())

## Step 2 — the correlator fingerprints the friend

Two events from the same IP+share would now merge into ONE
campaign. Let's prove it: send a follow-up `port_scan` from the
same egress with a different UA — they must still correlate.

In [ ]:
r2 = c.post(
    "/mesh/attack",
    headers={"x-forwarded-for": xff, "user-agent": "nmap/7.92", "x-aegis-peer": "friend-007"},
    json={"service": "http", "src_ip": xff, "payload": "nmap -sS", "user_agent": "nmap/7.92"},
)
show("second event, different UA", r2.json())

## Step 3 — campaign state snapshot

Both events should be in a single active campaign. The correlator
tracks severity per service (SSH > SMB > HTTP weights) and emits
a stream entry + SSE event every ingest.

In [ ]:
state = c.get("/mesh/state").json()
show("mesh state", {
    "campaigns": state["campaigns"],
    "bounty": state.get("bounty", {}),
})

## Step 4 — fire a countermeasure

The operator hits `POST /mesh/countermeasure` with the campaign_id.
This triggers the QAOA-inspired knapsack solver, mints an ML-KEM
decoy token, and awards the peer a bounty. The campaign must be
attributed to a peer handle for the bounty to mint.

In [ ]:
campaign_id = state["campaigns"][0]["campaign_id"]
cm = c.post(
    "/mesh/countermeasure",
    json={"campaign_id": campaign_id, "block_upstream": True, "rotate_keys": False},
).json()
show("countermeasure deployed", cm)

# Bounty should be in the response if the campaign was attributed.
bounty = cm.get("bounty")
if bounty:
    print(f"\n*** BOUNTY AWARDED: {bounty['amount']} QShield to {bounty['peer']} ***")

## Step 5 — peer claims the bounty

The friend runs `POST /mesh/bounty/claim` with the HMAC token
the operator showed them. Constant-time HMAC compare; one-shot
use.

In [ ]:
if bounty:
    claim = c.post(
        "/mesh/bounty/claim",
        json={
            "peer": bounty["peer"],
            "campaign_id": bounty["campaign_id"],
            "token": bounty["token"],
        },
    ).json()
    show("bounty claim", claim)
else:
    print("(no bounty — campaign was not attributed to a peer handle)")

## Step 6 — ledger audit

The whole chain above — share issued, attack ingested, campaign
correlated, countermeasure deployed, bounty awarded, bounty
claimed — is appended to the hash-chained ledger. We can verify
the chain integrity:

In [ ]:
verify = c.get("/ledger/verify").json()
show("ledger verify", verify)